# Notebook 02 · Pemodelan

| | |
|---|---|
| Nama | Dimas Fadilah Permana R |
| NIM | 25.55.1752 |
| Program Studi | S2 PJJ Informatika, Universitas AMIKOM Yogyakarta |
| Pembimbing | Prof. Dr. Ema Utami, S.Si., M.Kom. |


### Pertanyaan penelitian yang dijawab

1. Seberapa besar transkrip ASR menjelaskan retensi tontonan setelah efek durasi dikendalikan?
2. Dengan input yang sama, apakah BERT dan Bi-LSTM lebih baik daripada TF-IDF dengan model klasik?
3. Mengapa penambahan durasi membantu Bi-LSTM tetapi tidak membantu BERT?

### Desain model

Model disusun dalam tiga jalur input supaya perbandingannya setara. Model hanya dibandingkan dengan
model lain yang menerima input yang sama.

| Kode | Representasi teks | Algoritma | Input | Peran |
|---|---|---|---|---|
| `durasi_xgb` | (tidak ada) | XGBoost | durasi | Pembanding: seberapa jauh durasi saja |
| `tfidf_rf_teks` | TF-IDF | Random Forest | teks | Jalur teks saja |
| `tfidf_xgb_teks` | TF-IDF | XGBoost | teks | Jalur teks saja |
| `bilstm_teks` | Embedding dilatih dari nol | Bi-LSTM | teks | Jalur teks saja (kondisi A) |
| `bert_teks` | BERT pra-latih | BERT + head regresi | teks | Jalur teks saja (kondisi C) |
| `tfidf_rf_gab` | TF-IDF | Random Forest | teks + durasi | Jalur gabungan |
| `tfidf_xgb_gab` | TF-IDF | XGBoost | teks + durasi | Jalur gabungan |
| `bilstm_gab` | Embedding dilatih dari nol | Bi-LSTM | teks + durasi | Jalur gabungan (kondisi B) |
| `bert_gab` | BERT pra-latih | BERT + head regresi | teks + durasi | Jalur gabungan (kondisi D) |
| `bert_gab_proyeksi` | BERT pra-latih | BERT, durasi diproyeksikan dulu | teks + durasi | Ablation cara memasukkan durasi |
| `bert_gab_token` | BERT pra-latih | BERT, durasi ditulis di awal teks | teks + durasi | Ablation cara memasukkan durasi |

### Eksperimen yang dijalankan

| Kode | Partisi | Target | Model | Tujuan |
|---|---|---|---|---|
| `utama` | acak | `retensi` | semua, termasuk ablation | Eksperimen utama |
| `adj` | acak | `retensi_adj` | semua kecuali ablation | Sinyal teks setelah efek durasi dikeluarkan |
| `author` | per kreator | `retensi` | semua kecuali ablation | Ketahanan pada kreator yang belum pernah dilihat |
| `noclip` | acak | `retensi_tanpa_clip` | model klasik saja | Sensitivitas terhadap pemotongan rasio |


---
# Persiapan Environment
### Deteksi Environment (Kaggle, Colab, atau laptop)

In [1]:
import os

# Kaggle juga memasang paket google.colab, jadi Colab dikenali dari penanda khas Colab
DI_KAGGLE = os.path.exists('/kaggle/input') or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
IN_COLAB = (not DI_KAGGLE) and (os.path.exists('/var/colab/hostname') or 'COLAB_RELEASE_TAG' in os.environ)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
LINGKUNGAN = 'Kaggle' if DI_KAGGLE else 'Colab' if IN_COLAB else 'Lokal'
print(f"Lingkungan: {LINGKUNGAN}")

Lingkungan: Kaggle


### Impor pustaka
`scikit-learn` dan `xgboost` untuk model klasik, `torch` untuk Bi-LSTM, dan `transformers` untuk BERT.
Semuanya sudah tersedia di Kaggle dan Colab.

In [2]:
import gc, json, math, random, time, warnings
from collections import Counter

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import spearmanr

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from transformers import BertTokenizerFast, BertModel, logging as hf_logging

warnings.filterwarnings('ignore')
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | perangkat: {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == 'cuda' else ''))
if DEVICE.type != 'cuda':
    print("Catatan: tanpa GPU, pelatihan BERT sangat lambat. Kaggle: Settings > Accelerator > GPU T4. "
          "Colab: Runtime > Change runtime type > GPU.")

PyTorch 2.11.0+cu128 | perangkat: cuda (Tesla T4)


### Konfigurasi

Semua pengaturan eksperimen ada di sel ini.

**Pengaturan yang sama untuk semua model deep learning:** optimizer AdamW, loss MSE, maksimal 8
epoch, batch 32, *early stopping* bila RMSE validasi tidak membaik 2 epoch berturut-turut, panjang
teks maksimal 128 token, dropout 0,3, dan *head* regresi yang sama (Dense 64, ReLU, Dropout, Dense 1).

**Pencarian hiperparameter.** Setiap model mendapat grid kecil yang sebanding (4 sampai 6
kombinasi untuk model klasik, 3 learning rate untuk model deep learning). Semua kombinasi dilatih
pada data latih, lalu **kombinasi dengan RMSE validasi terendah** dipilih. Data uji hanya dipakai
satu kali, setelah pemilihan selesai.

In [3]:
import fnmatch

def cari_berkas(akar, pola):
    # telusuri semua subfolder (termasuk symlink), kembalikan berkas pertama yang cocok dengan pola
    for folder, subfolder, berkas in os.walk(akar, followlinks=True):
        subfolder.sort()
        for nama in sorted(berkas):
            if fnmatch.fnmatch(nama, pola):
                return os.path.join(folder, nama)
    return None

def cari_folder(akar, nama_folder, isi=None):
    # folder bernama nama_folder (dan berisi 'isi', bila diberikan)
    for folder, subfolder, berkas in os.walk(akar, followlinks=True):
        subfolder.sort()
        if os.path.basename(folder) == nama_folder and (isi is None or isi in subfolder + berkas):
            return folder
    return None

def tampilkan_isi(akar, kedalaman=3):
    # dipakai bila ada berkas yang tidak ditemukan, untuk melihat struktur folder input
    print(f"Isi {akar}:")
    for folder, subfolder, berkas in os.walk(akar, followlinks=True):
        tingkat = folder[len(akar):].count(os.sep)
        print('   ' * tingkat + (os.path.basename(folder) or akar) + f'/  ({len(berkas)} berkas)')
        if tingkat >= kedalaman - 1:
            subfolder[:] = []
        for b in sorted(berkas)[:4]:
            print('   ' * (tingkat + 1) + b)

SEED = 42
DEBUG = os.environ.get('TESIS_DEBUG', '0') == '1'    # uji cepat di laptop
RESUME = True                                         # lewati model yang hasilnya sudah tersimpan

if DI_KAGGLE:
    # keluaran notebook 01 dicari otomatis di /kaggle/input; hasil ditulis ke /kaggle/working
    DATA_DIR = {'random': cari_folder('/kaggle/input', 'final_revisi', 'train.csv'),
                'author': cari_folder('/kaggle/input', 'final_revisi_authorsplit', 'train.csv')}
    if not all(DATA_DIR.values()):
        tampilkan_isi('/kaggle/input')
        raise FileNotFoundError("Keluaran notebook 01 belum ditemukan (Add Input > Your Work > notebook 01).")
    OUT_BASE = '/kaggle/working'
else:
    if IN_COLAB:
        OUT_BASE = '/content/drive/MyDrive/Tesis_Dimas/dataset_tsinghua'
    else:
        OUT_BASE = os.environ.get('TESIS_OUT_BASE', './hasil')
    DATA_DIR = {'random': f'{OUT_BASE}/final_revisi', 'author': f'{OUT_BASE}/final_revisi_authorsplit'}
HASIL_DIR = f'{OUT_BASE}/hasil_model_revisi' + ('_debug' if DEBUG else '')
BERT_MODEL_NAME = os.environ.get('TESIS_BERT_MODEL', 'bert-base-uncased')

EKSPERIMEN = [
    {'kode': 'utama',  'split': 'random', 'target': 'retensi',            'deep': True,  'ablation': True},
    {'kode': 'adj',    'split': 'random', 'target': 'retensi_adj',        'deep': True,  'ablation': False},
    {'kode': 'author', 'split': 'author', 'target': 'retensi',            'deep': True,  'ablation': False},
    {'kode': 'noclip', 'split': 'random', 'target': 'retensi_tanpa_clip', 'deep': False, 'ablation': False},
]

# Model klasik
TFIDF_PARAM = dict(max_features=5000, ngram_range=(1, 2), min_df=2)
GRID_DURASI = {'max_depth': [2, 3, 4], 'learning_rate': [0.05, 0.1]}
GRID_RF = {'max_features': ['sqrt', 0.1], 'min_samples_leaf': [1, 5]}
GRID_XGB = {'max_depth': [3, 6], 'learning_rate': [0.05, 0.1]}
RF_POHON, XGB_POHON_MAKS, XGB_EARLY_STOP = 200, 1000, 30

# Model deep learning
MAX_LEN, EPOCHS, BATCH_SIZE, PATIENCE = 128, 8, 32, 2
EMBED_DIM, HIDDEN, DROPOUT = 100, 128, 0.3
LR_LSTM = [1e-3, 3e-3, 5e-3]
LR_BERT = [1e-5, 3e-5, 5e-5]

SUBSET = None
if DEBUG:   # hanya untuk memastikan kode jalan, bukan untuk hasil
    EPOCHS, LR_LSTM, LR_BERT, SUBSET = 1, [3e-3], [3e-5], 300
    RF_POHON, XGB_POHON_MAKS = 30, 100
    GRID_DURASI, GRID_XGB, GRID_RF = {'max_depth': [3]}, {'max_depth': [3]}, {'max_features': ['sqrt']}

os.makedirs(HASIL_DIR, exist_ok=True)
print(f"Mode          : {'DEBUG (uji kode)' if DEBUG else 'PENUH'}")
print(f"Data          : {DATA_DIR['random']}")
print(f"Hasil         : {HASIL_DIR}")
print(f"Model BERT    : {BERT_MODEL_NAME}")
print(f"Eksperimen    : {[e['kode'] for e in EKSPERIMEN]}")

Mode          : PENUH
Data          : /kaggle/input/notebooks/dimasfadilahamikom/notebook-01-preprocessing/final_revisi
Hasil         : /kaggle/working/hasil_model_revisi
Model BERT    : bert-base-uncased
Eksperimen    : ['utama', 'adj', 'author', 'noclip']


### Catatan metodologis: grid learning rate BERT dan Bi-LSTM dibedakan

BERT tidak dilatih dari nol. Bobot pra-latihnya hanya disesuaikan (*fine-tuning*), sehingga learning
rate kecil {1e-5, 3e-5, 5e-5} adalah rentang yang dianjurkan (Devlin dkk., 2019). Bi-LSTM dalam
penelitian ini dilatih dari nol, termasuk embedding-nya. Dengan learning rate sekecil itu, Bi-LSTM
tidak sempat konvergen dalam 8 epoch, sehingga perbandingannya justru tidak adil. Karena itu
Bi-LSTM memakai rentang {1e-3, 3e-3, 5e-3}.

Yang dijaga sama bukan angka learning rate-nya, tetapi prosedurnya: jumlah nilai yang dicoba,
jumlah epoch, batch, loss, optimizer, *head* regresi, dan cara memilih kombinasi terbaik.

### Environment Locking

In [4]:
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
print(f"Seed dikunci ke {SEED}.")

Seed dikunci ke 42.


---
# TAHAP 1 · Data dan Fungsi Bersama
### 1.1 Memuat data satu eksperimen

Fungsi `siapkan_eksperimen` membaca partisi latih, validasi, dan uji dari notebook 01, mengambil
target yang dipilih, lalu menyiapkan folder hasil. Bila sebagian model sudah pernah dilatih,
prediksinya dimuat agar tidak dilatih ulang.

| Peran | Kolom |
|---|---|
| X teks | `text_clean` |
| X durasi | `duration_norm` (durasi setelah log dan MinMax) |
| Y | `retensi`, `retensi_adj`, atau `retensi_tanpa_clip` |

In [5]:
def siapkan_eksperimen(cfg):
    folder = DATA_DIR[cfg['split']]
    bagian = {}
    for nama in ['train', 'val', 'test']:
        d = pd.read_csv(f'{folder}/{nama}.csv')
        d['text_clean'] = d['text_clean'].fillna('').astype(str)
        if SUBSET:
            d = d.sample(min(SUBSET if nama == 'train' else SUBSET // 3, len(d)), random_state=SEED)
        bagian[nama] = d.reset_index(drop=True)

    E = dict(cfg=cfg, **bagian)
    E['y'] = {k: v[cfg['target']].values.astype(np.float32) for k, v in bagian.items()}
    E['durasi'] = {k: v[['duration_norm']].values.astype(np.float32) for k, v in bagian.items()}
    E['folder'] = f"{HASIL_DIR}/{cfg['kode']}"
    os.makedirs(E['folder'], exist_ok=True)

    path_pred = f"{E['folder']}/prediksi_uji.csv"
    if RESUME and os.path.exists(path_pred):
        E['prediksi'] = pd.read_csv(path_pred)
    else:
        E['prediksi'] = pd.DataFrame({'pid': bagian['test']['pid'], 'y_true': E['y']['test']})
    path_log = f"{E['folder']}/log_tuning.csv"
    E['log'] = pd.read_csv(path_log) if (RESUME and os.path.exists(path_log)) else pd.DataFrame()
    with open(f"{E['folder']}/pengaturan.json", 'w') as f:
        json.dump({**cfg, 'bert_model': BERT_MODEL_NAME, 'epochs': EPOCHS, 'batch': BATCH_SIZE,
                   'max_len': MAX_LEN, 'lr_bert': LR_BERT, 'lr_lstm': LR_LSTM, 'seed': SEED}, f, indent=2)
    print(f"[{cfg['kode']}] partisi {cfg['split']}, target {cfg['target']} | "
          f"latih {len(E['train']):,} | validasi {len(E['val']):,} | uji {len(E['test']):,}")
    return E

### 1.2 Mencatat dan menyimpan hasil

`jalankan` adalah pembungkus untuk setiap model. Fungsi ini melewati model yang hasilnya sudah ada,
menjalankan pencarian hiperparameter.

In [6]:
def rmse(y, p):
    return math.sqrt(mean_squared_error(y, p))

def jalankan(E, kode_model, fungsi):
    if RESUME and kode_model in E['prediksi'].columns:
        print(f"  {kode_model:20s} sudah ada, dilewati.")
        return
    mulai = time.time()
    pred_uji, log = fungsi(E)
    E['prediksi'][kode_model] = np.asarray(pred_uji, dtype=float).ravel()
    log = pd.DataFrame(log).assign(model=kode_model)
    log['terpilih'] = log['rmse_val'] == log['rmse_val'].min()
    E['log'] = pd.concat([E['log'], log], ignore_index=True)
    E['prediksi'].to_csv(f"{E['folder']}/prediksi_uji.csv", index=False)
    E['log'].to_csv(f"{E['folder']}/log_tuning.csv", index=False)
    terbaik = log[log['terpilih']].iloc[0]
    src = spearmanr(E['y']['test'], E['prediksi'][kode_model])[0]
    print(f"  {kode_model:20s} terpilih {terbaik['parameter']:32s} RMSE val {terbaik['rmse_val']:.4f} "
          f"| SRC uji {src:.3f} | {time.time() - mulai:.0f} dtk")

### 1.3 Memuat data eksperimen utama
Tahap 2 sampai 4 menampilkan pelatihan setiap model pada eksperimen utama. Eksperimen lain
dijalankan dengan fungsi yang sama di Tahap 5.

In [7]:
E_UTAMA = siapkan_eksperimen(EKSPERIMEN[0])
display(E_UTAMA['train'][['pid', 'text_clean', 'duration', 'duration_norm', 'retensi']].head()
        .assign(text_clean=lambda d: d['text_clean'].str.slice(0, 70) + '...'))

[utama] partisi random, target retensi | latih 3,253 | validasi 697 | uji 698


,pid,text_clean,duration,duration_norm,retensi
0,33393,how to get the blue sky when it's not there? j...,15.133,0.329369,0.673032
1,10514,"go to the south and see, go to dongguan and se...",49.500,0.709353,0.407828
2,37407,if you accidentally swallow a piece of chewing...,36.800,0.612896,0.432609
3,146537,hello! how can i assist you today?...,10.500,0.216643,0.757576
4,61364,"oh, it's you again, don't disturb me, i'm sunb...",47.040,0.692723,0.508765


---
# TAHAP 2 · Model Klasik

### 2.1 Pencarian grid untuk model klasik
Setiap kombinasi pada grid dilatih di data latih dan dinilai dengan RMSE pada data validasi.
Untuk XGBoost, jumlah pohon ditentukan oleh *early stopping* pada data validasi (maksimal 1.000
pohon, berhenti bila 30 pohon berturut-turut tidak memperbaiki validasi).

In [8]:
def cari_grid(buat_model, grid, X, y, pakai_eval=False):
    log, terbaik = [], None
    for param in ParameterGrid(grid):
        model = buat_model(**param)
        if pakai_eval:
            model.fit(X['train'], y['train'], eval_set=[(X['val'], y['val'])], verbose=False)
            param = {**param, 'n_pohon': model.best_iteration + 1}
        else:
            model.fit(X['train'], y['train'])
        skor = rmse(y['val'], model.predict(X['val']))
        log.append({'parameter': ', '.join(f'{k}={v}' for k, v in param.items()), 'rmse_val': skor})
        if terbaik is None or skor < terbaik[0]:
            terbaik = (skor, model)
    return terbaik[1].predict(X['test']), log

def buat_xgb(**p):
    return XGBRegressor(n_estimators=XGB_POHON_MAKS, early_stopping_rounds=XGB_EARLY_STOP,
                        subsample=0.9, colsample_bytree=0.9, objective='reg:squarederror',
                        random_state=SEED, n_jobs=-1, **p)

def buat_rf(**p):
    return RandomForestRegressor(n_estimators=RF_POHON, random_state=SEED, n_jobs=-1, **p)

### 2.2 Pembanding XGBoost dengan durasi saja
Model ini tidak membaca teks sama sekali. Hasilnya menjadi patokan model yang memakai durasi
baru bisa dikatakan memanfaatkan teks bila hasilnya melebihi model ini.

In [9]:
def model_durasi_xgb(E):
    return cari_grid(buat_xgb, GRID_DURASI, E['durasi'], E['y'], pakai_eval=True)

jalankan(E_UTAMA, 'durasi_xgb', model_durasi_xgb)

  durasi_xgb           terpilih learning_rate=0.05, max_depth=2, n_pohon=45 RMSE val 0.1924 | SRC uji 0.476 | 1 dtk


### 2.3 Fitur TF-IDF
TF-IDF memberi bobot tinggi pada kata yang sering muncul di satu transkrip tetapi jarang di transkrip
lain. Vectorizer di-*fit* hanya pada teks latih (unigram dan bigram, 5.000 istilah, istilah harus
muncul minimal di 2 transkrip). Untuk jalur gabungan, kolom `duration_norm` disambungkan ke matriks
TF-IDF sebagai satu kolom tambahan.

In [10]:
def fitur_tfidf(E, pakai_durasi):
    if 'tfidf' not in E:
        vec = TfidfVectorizer(**TFIDF_PARAM)
        E['tfidf'] = {'train': vec.fit_transform(E['train']['text_clean'])}
        E['tfidf'].update({k: vec.transform(E[k]['text_clean']) for k in ['val', 'test']})
        E['tfidf_vocab'] = len(vec.vocabulary_)
    if not pakai_durasi:
        return E['tfidf']
    return {k: sparse.hstack([E['tfidf'][k], sparse.csr_matrix(E['durasi'][k])]).tocsr()
            for k in ['train', 'val', 'test']}

X = fitur_tfidf(E_UTAMA, pakai_durasi=True)
print(f"Istilah TF-IDF: {E_UTAMA['tfidf_vocab']:,} | dimensi jalur gabungan: {X['train'].shape}")

Istilah TF-IDF: 5,000 | dimensi jalur gabungan: (3253, 5001)


### 2.4 Random Forest dan XGBoost, jalur teks saja dan jalur gabungan
Random Forest merata-ratakan banyak pohon keputusan yang masing-masing dilatih pada sampel data dan
fitur yang berbeda. XGBoost membangun pohon secara bertahap, dan setiap pohon baru memperbaiki galat
pohon sebelumnya.

In [11]:
def model_tfidf(algoritma, pakai_durasi):
    def latih(E):
        X = fitur_tfidf(E, pakai_durasi)
        if algoritma == 'rf':
            return cari_grid(buat_rf, GRID_RF, X, E['y'])
        return cari_grid(buat_xgb, GRID_XGB, X, E['y'], pakai_eval=True)
    return latih

for algoritma in ['rf', 'xgb']:
    for pakai_durasi, jalur in [(False, 'teks'), (True, 'gab')]:
        jalankan(E_UTAMA, f'tfidf_{algoritma}_{jalur}', model_tfidf(algoritma, pakai_durasi))

  tfidf_rf_teks        terpilih max_features=0.1, min_samples_leaf=5 RMSE val 0.2030 | SRC uji 0.299 | 28 dtk
  tfidf_rf_gab         terpilih max_features=0.1, min_samples_leaf=1 RMSE val 0.1884 | SRC uji 0.461 | 29 dtk
  tfidf_xgb_teks       terpilih learning_rate=0.05, max_depth=3, n_pohon=138 RMSE val 0.2023 | SRC uji 0.277 | 40 dtk
  tfidf_xgb_gab        terpilih learning_rate=0.05, max_depth=3, n_pohon=165 RMSE val 0.1888 | SRC uji 0.483 | 36 dtk


---
# TAHAP 3 · Prosedur Pelatihan Deep Learning (Bersama)

Bi-LSTM dan BERT dilatih dengan fungsi yang sama persis, supaya perbedaan hasil berasal dari
arsitekturnya, bukan dari cara melatihnya.

* `latih_torch`: AdamW, loss MSE, evaluasi RMSE validasi tiap epoch, *early stopping* dengan
 2 epoch, dan bobot terbaik dikembalikan di akhir.
* `cari_lr`: mencoba setiap learning rate pada grid, memilih yang RMSE validasinya terendah, lalu
  memprediksi data uji dengan model terpilih.

In [12]:
@torch.no_grad()
def prediksi_torch(model, ds):
    model.eval()
    hasil = []
    for batch in DataLoader(ds, batch_size=BATCH_SIZE * 2):
        batch = [b.to(DEVICE) for b in batch]
        hasil.append(model(*batch[:-1]).float().cpu().numpy())
    return np.concatenate(hasil)


def latih_torch(buat_model, ds_latih, ds_val, lr):
    set_seed()
    model = buat_model().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    loader = DataLoader(ds_latih, batch_size=BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    y_val = ds_val.tensors[-1].numpy()
    terbaik, bobot_terbaik, sabar, riwayat = float('inf'), None, 0, []
    for epoch in range(EPOCHS):
        model.train()
        for batch in loader:
            batch = [b.to(DEVICE) for b in batch]
            optimizer.zero_grad()
            loss = loss_fn(model(*batch[:-1]), batch[-1])
            loss.backward()
            optimizer.step()
        skor = rmse(y_val, prediksi_torch(model, ds_val))
        riwayat.append(round(skor, 4))
        if skor < terbaik - 1e-5:
            terbaik, sabar = skor, 0
            bobot_terbaik = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            sabar += 1
            if sabar >= PATIENCE:
                break
    model.load_state_dict(bobot_terbaik)
    return model, terbaik, riwayat


def cari_lr(buat_model, ds, grid):
    log, terbaik = [], None
    for lr in grid:
        model, skor, riwayat = latih_torch(buat_model, ds['train'], ds['val'], lr)
        log.append({'parameter': f'lr={lr:g}, epoch={len(riwayat)}', 'rmse_val': skor,
                    'riwayat_rmse_val': str(riwayat)})
        if terbaik is None or skor < terbaik[0]:
            terbaik = (skor, model)
        else:
            del model
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    pred = prediksi_torch(terbaik[1], ds['test'])
    del terbaik
    return pred, log

---
# TAHAP 4 · Bi-LSTM (Kondisi A dan B)

Bi-LSTM membaca transkrip dari dua arah, kiri ke kanan dan kanan ke kiri, lalu menggabungkan
keadaan akhir kedua arah menjadi satu vektor teks 256 dimensi.

### 4.1 Vocabulary dan urutan indeks
Vocabulary dibangun dari teks latih saja (kata yang muncul minimal 2 kali). Indeks 0 untuk
`<pad>` dan 1 untuk kata yang tidak dikenal. Setiap transkrip diubah menjadi urutan indeks
sepanjang 128, dan panjang aslinya disimpan.

In [13]:
def data_bilstm(E):
    if 'seq' in E:
        return E['seq']
    hitung = Counter(w for t in E['train']['text_clean'] for w in t.split())
    kata = ['<pad>', '<unk>'] + [w for w, c in hitung.most_common(30000) if c >= 2]
    indeks = {w: i for i, w in enumerate(kata)}
    E['vocab_size'] = len(kata)

    def ubah(teks):
        ids = [indeks.get(w, 1) for w in teks.split()[:MAX_LEN]]
        return ids + [0] * (MAX_LEN - len(ids)), max(len(ids), 1)

    E['seq'] = {}
    for k in ['train', 'val', 'test']:
        ids, panjang = zip(*[ubah(t) for t in E[k]['text_clean']])
        E['seq'][k] = TensorDataset(torch.tensor(ids), torch.tensor(panjang),
                                    torch.tensor(E['durasi'][k]), torch.tensor(E['y'][k]))
    return E['seq']

ds = data_bilstm(E_UTAMA)
print(f"Ukuran vocabulary: {E_UTAMA['vocab_size']:,} kata")

Ukuran vocabulary: 10,699 kata


### 4.2 Arsitektur
1. **Embedding** 100 dimensi, dilatih dari nol bersama model.
2. **Bi-LSTM** 128 unit per arah. Urutan diproses dengan *packed sequence*, sehingga posisi
   `<pad>` tidak ikut dibaca dan keadaan akhir benar-benar berasal dari kata terakhir transkrip.
3. **Late fusion** (kondisi B): vektor teks 256 dimensi disambung dengan durasi.
4. **Head regresi**: Dense 64, ReLU, Dropout 0,3, Dense 1.

In [14]:
class BiLSTMRegressor(nn.Module):
    def __init__(self, vocab_size, pakai_durasi):
        super().__init__()
        self.pakai_durasi = pakai_durasi
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Linear(HIDDEN * 2 + int(pakai_durasi), 64), nn.ReLU(),
                                  nn.Dropout(DROPOUT), nn.Linear(64, 1))

    def forward(self, ids, panjang, durasi):
        x = nn.utils.rnn.pack_padded_sequence(self.embedding(ids), panjang.cpu(),
                                              batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(x)
        vektor = torch.cat([h[-2], h[-1]], dim=1)          # arah maju dan mundur
        if self.pakai_durasi:
            vektor = torch.cat([vektor, durasi], dim=1)     # late fusion
        return self.head(vektor).squeeze(1)

n_param = sum(p.numel() for p in BiLSTMRegressor(E_UTAMA['vocab_size'], True).parameters())
print(f"Jumlah parameter Bi-LSTM: {n_param:,}")

Jumlah parameter Bi-LSTM: 1,321,997


### 4.3 Pelatihan kondisi A (teks) dan B (teks + durasi)

In [15]:
def model_bilstm(pakai_durasi):
    def latih(E):
        ds = data_bilstm(E)
        return cari_lr(lambda: BiLSTMRegressor(E['vocab_size'], pakai_durasi), ds, LR_LSTM)
    return latih

jalankan(E_UTAMA, 'bilstm_teks', model_bilstm(False))
jalankan(E_UTAMA, 'bilstm_gab', model_bilstm(True))

  bilstm_teks          terpilih lr=0.005, epoch=3                RMSE val 0.2244 | SRC uji 0.100 | 15 dtk
  bilstm_gab           terpilih lr=0.003, epoch=3                RMSE val 0.2126 | SRC uji 0.255 | 16 dtk


---
# TAHAP 5 · BERT (Kondisi C dan D, plus Ablation Durasi)

BERT (`bert-base-uncased`, 12 lapisan, 768 dimensi, sekitar 110 juta parameter) sudah dilatih
sebelumnya pada korpus besar, lalu disesuaikan (*fine-tuning*) untuk memprediksi retensi.
Representasi transkrip diambil dari token `[CLS]` pada lapisan terakhir.

### 5.1 Tokenisasi
Tokenizer WordPiece memecah kata yang tidak dikenal menjadi potongan subkata, sehingga tidak ada
kata yang hilang. Panjang maksimal 128 token.

Untuk ablation `bert_gab_token`, durasi ditulis sebagai teks di awal transkrip, misalnya
`"duration 23 seconds. i am a brave..."`, sehingga BERT membaca durasi lewat jalur yang sama
dengan teks.

In [16]:
tokenizer = BertTokenizerFast.from_pretrained(BERT_MODEL_NAME)

def data_bert(E, durasi_sebagai_teks=False):
    kunci = 'bert_token' if durasi_sebagai_teks else 'bert'
    if kunci in E:
        return E[kunci]
    E[kunci] = {}
    for k in ['train', 'val', 'test']:
        teks = E[k]['text_clean']
        if durasi_sebagai_teks:
            teks = 'duration ' + E[k]['duration'].round().astype(int).astype(str) + ' seconds. ' + teks
        enc = tokenizer(list(teks), max_length=MAX_LEN, padding='max_length', truncation=True,
                        return_tensors='pt')
        E[kunci][k] = TensorDataset(enc['input_ids'], enc['attention_mask'],
                                    torch.tensor(E['durasi'][k]), torch.tensor(E['y'][k]))
    return E[kunci]

ds = data_bert(E_UTAMA)
contoh = ds['train'].tensors[0][0]
print(f"Model: {BERT_MODEL_NAME} | vocabulary {tokenizer.vocab_size:,} token")
print("Contoh token:", tokenizer.convert_ids_to_tokens(contoh[:20]))

Model: bert-base-uncased | vocabulary 30,522 token
Contoh token: ['[CLS]', 'how', 'to', 'get', 'the', 'blue', 'sky', 'when', 'it', "'", 's', 'not', 'there', '?', 'just', 'buy', 'a', 'new', 'one', ',']


### 5.2 Arsitektur dan tiga cara memasukkan durasi

| Kode | Cara durasi masuk ke BERT | Alasan diuji |
|---|---|---|
| `bert_teks` | Tidak dipakai | Kondisi C |
| `bert_gab` | Satu angka disambung ke vektor `[CLS]` 768 dimensi (late fusion) | Kondisi D, sama dengan Bi-LSTM |
| `bert_gab_proyeksi` | Durasi diproyeksikan dulu menjadi vektor 32 dimensi (Dense, ReLU), baru disambung | Menguji apakah satu angka "tenggelam" di antara 768 angka lain |
| `bert_gab_token` | Durasi ditulis di awal transkrip | Menguji apakah durasi lebih terpakai bila dibaca sebagai teks |

Ablation ini menjawab komentar Reviewer 2 bahwa kegagalan BERT memakai durasi bisa disebabkan
skala fitur atau arsitektur, bukan hanya dimensi representasi.

In [17]:
class BertRegressor(nn.Module):
    def __init__(self, cara_durasi):     # 'tidak', 'late', atau 'proyeksi'
        super().__init__()
        self.bert = BertModel.from_pretrained(BERT_MODEL_NAME)
        self.cara_durasi = cara_durasi
        dim_durasi = {'tidak': 0, 'late': 1, 'proyeksi': 32}[cara_durasi]
        if cara_durasi == 'proyeksi':
            self.proyeksi = nn.Sequential(nn.Linear(1, 32), nn.ReLU())
        self.dropout = nn.Dropout(DROPOUT)
        self.head = nn.Sequential(nn.Linear(self.bert.config.hidden_size + dim_durasi, 64), nn.ReLU(),
                                  nn.Dropout(DROPOUT), nn.Linear(64, 1))

    def forward(self, ids, mask, durasi):
        vektor = self.dropout(self.bert(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0])
        if self.cara_durasi == 'late':
            vektor = torch.cat([vektor, durasi], dim=1)
        elif self.cara_durasi == 'proyeksi':
            vektor = torch.cat([vektor, self.proyeksi(durasi)], dim=1)
        return self.head(vektor).squeeze(1)

n_param = sum(p.numel() for p in BertRegressor('late').parameters())
print(f"Jumlah parameter BERT + head: {n_param:,}")

Jumlah parameter BERT + head: 109,531,585


### 5.3 Pelatihan kondisi C, D, dan ablation
Bagian ini yang paling lama. Setiap varian dilatih dengan 3 learning rate.

In [18]:
def model_bert(cara_durasi, durasi_sebagai_teks=False):
    def latih(E):
        ds = data_bert(E, durasi_sebagai_teks)
        return cari_lr(lambda: BertRegressor(cara_durasi), ds, LR_BERT)
    return latih

VARIAN_BERT = {
    'bert_teks': model_bert('tidak'),
    'bert_gab': model_bert('late'),
    'bert_gab_proyeksi': model_bert('proyeksi'),
    'bert_gab_token': model_bert('tidak', durasi_sebagai_teks=True),
}
for kode_model, fungsi in VARIAN_BERT.items():
    jalankan(E_UTAMA, kode_model, fungsi)

  bert_teks            terpilih lr=1e-05, epoch=5                RMSE val 0.2017 | SRC uji 0.272 | 778 dtk
  bert_gab             terpilih lr=5e-05, epoch=5                RMSE val 0.2006 | SRC uji 0.319 | 917 dtk
  bert_gab_proyeksi    terpilih lr=1e-05, epoch=5                RMSE val 0.2025 | SRC uji 0.302 | 1057 dtk
  bert_gab_token       terpilih lr=5e-05, epoch=4                RMSE val 0.1907 | SRC uji 0.470 | 990 dtk


---
# TAHAP 6 · Eksperimen Lain

Eksperimen `adj`, `author`, dan `noclip` memakai fungsi yang sama persis dengan Tahap 2 sampai 5.

In [19]:
MODEL_KLASIK = {'durasi_xgb': model_durasi_xgb,
                **{f'tfidf_{a}_{j}': model_tfidf(a, d) for a in ['rf', 'xgb'] for d, j in [(False, 'teks'), (True, 'gab')]}}
MODEL_DEEP = {'bilstm_teks': model_bilstm(False), 'bilstm_gab': model_bilstm(True),
              'bert_teks': VARIAN_BERT['bert_teks'], 'bert_gab': VARIAN_BERT['bert_gab']}
MODEL_ABLATION = {k: VARIAN_BERT[k] for k in ['bert_gab_proyeksi', 'bert_gab_token']}

def jalankan_eksperimen(cfg):
    E = siapkan_eksperimen(cfg)
    daftar = dict(MODEL_KLASIK)
    if cfg['deep']:
        daftar.update(MODEL_DEEP)
    if cfg['ablation']:
        daftar.update(MODEL_ABLATION)
    for kode_model, fungsi in daftar.items():
        jalankan(E, kode_model, fungsi)
    return E

for cfg in EKSPERIMEN[1:]:
    print()
    jalankan_eksperimen(cfg)


[adj] partisi random, target retensi_adj | latih 3,253 | validasi 697 | uji 698
  durasi_xgb           terpilih learning_rate=0.1, max_depth=3, n_pohon=21 RMSE val 0.1922 | SRC uji 0.014 | 0 dtk
  tfidf_rf_teks        terpilih max_features=0.1, min_samples_leaf=5 RMSE val 0.1886 | SRC uji 0.092 | 37 dtk
  tfidf_rf_gab         terpilih max_features=0.1, min_samples_leaf=5 RMSE val 0.1887 | SRC uji 0.098 | 37 dtk
  tfidf_xgb_teks       terpilih learning_rate=0.1, max_depth=6, n_pohon=17 RMSE val 0.1891 | SRC uji 0.110 | 28 dtk
  tfidf_xgb_gab        terpilih learning_rate=0.05, max_depth=3, n_pohon=133 RMSE val 0.1885 | SRC uji 0.084 | 33 dtk
  bilstm_teks          terpilih lr=0.005, epoch=3                RMSE val 0.1926 | SRC uji 0.064 | 14 dtk
  bilstm_gab           terpilih lr=0.003, epoch=3                RMSE val 0.1934 | SRC uji 0.054 | 13 dtk
  bert_teks            terpilih lr=1e-05, epoch=8                RMSE val 0.1864 | SRC uji 0.182 | 1339 dtk
  bert_gab             terpili

---
# TAHAP 7 · Ringkasan Hasil Pelatihan

Tabel ini hanya pemeriksaan cepat bahwa semua model sudah selesai. Analisis lengkap (selang
kepercayaan, uji statistik, ablation, contoh prediksi) ada di notebook 03.

In [20]:
ringkasan = []
for cfg in EKSPERIMEN:
    folder = f"{HASIL_DIR}/{cfg['kode']}"
    pred = pd.read_csv(f'{folder}/prediksi_uji.csv')
    log = pd.read_csv(f'{folder}/log_tuning.csv')
    for kode_model in [c for c in pred.columns if c not in ('pid', 'y_true')]:
        pilih = log[(log['model'] == kode_model) & log['terpilih']].iloc[0]
        ringkasan.append({'eksperimen': cfg['kode'], 'model': kode_model,
                          'parameter terpilih': pilih['parameter'],
                          'RMSE val': round(pilih['rmse_val'], 4),
                          'SRC uji': round(spearmanr(pred['y_true'], pred[kode_model])[0], 3),
                          'MAE uji': round(mean_absolute_error(pred['y_true'], pred[kode_model]), 4)})
ringkasan = pd.DataFrame(ringkasan)
ringkasan.to_csv(f'{HASIL_DIR}/ringkasan_pelatihan.csv', index=False)
display(ringkasan[ringkasan['eksperimen'] == 'utama'])
print(f"\nSemua hasil tersimpan di {HASIL_DIR}")

,eksperimen,model,parameter terpilih,RMSE val,SRC uji,MAE uji
0,utama,durasi_xgb,"learning_rate=0.05, max_depth=2, n_pohon=45",0.1924,0.476,0.1484
1,utama,tfidf_rf_teks,"max_features=0.1, min_samples_leaf=5",0.2030,0.299,0.1644
2,utama,tfidf_rf_gab,"max_features=0.1, min_samples_leaf=1",0.1884,0.461,0.1508
3,utama,tfidf_xgb_teks,"learning_rate=0.05, max_depth=3, n_pohon=138",0.2023,0.277,0.1651
4,utama,tfidf_xgb_gab,"learning_rate=0.05, max_depth=3, n_pohon=165",0.1888,0.483,0.1473
5,utama,bilstm_teks,"lr=0.005, epoch=3",0.2244,0.100,0.1768
6,utama,bilstm_gab,"lr=0.003, epoch=3",0.2126,0.255,0.1658
7,utama,bert_teks,"lr=1e-05, epoch=5",0.2017,0.272,0.1648
8,utama,bert_gab,"lr=5e-05, epoch=5",0.2006,0.319,0.1615
9,utama,bert_gab_proyeksi,"lr=1e-05, epoch=5",0.2025,0.302,0.1629



Semua hasil tersimpan di /kaggle/working/hasil_model_revisi
